# Capstone Project - Rajendra Raju

Option 1 build using **Python, LangChain, and LangGraph**.

## Flow covered in this notebook
- document loading from the Day 5 capstone folders
- classification into **Cease / Uncertain / Irrelevant**
- routing to database, archive, or manual review
- audit logging for each processed document
- test mode for one / two / all PDFs

## Labels
- **Cease**
- **Uncertain**
- **Irrelevant**


## 1) Install Dependencies

Run this once in Google Colab before executing the remaining cells.


In [1]:
!pip -q install langchain langgraph langchain-groq pypdf pydantic typing-extensions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.3 MB/s eta 0:00:00


## 2) Load Keys from Google Colab Secrets

Create these secrets in Colab before running:
- `GROQ_API_KEY`
- `LANGSMITH_API_KEY`
- `ARIZE_API_KEY`
- `ARIZE_SPACE_ID`

`GROQ_API_KEY` is used for the core classification flow. The other keys are loaded when available.


In [2]:
import os

def load_secret(name: str, required: bool = False):
    value = None
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = os.environ.get(name)

    if value:
        os.environ[name] = value
        print(f"{name}: loaded")
        return value

    if required:
        raise ValueError(f"Missing required secret: {name}")

    print(f"{name}: not provided (optional)")
    return None

GROQ_API_KEY = load_secret("GROQ_API_KEY", required=True)
LANGSMITH_API_KEY = load_secret("LANGSMITH_API_KEY", required=False)
ARIZE_API_KEY = load_secret("ARIZE_API_KEY", required=False)
ARIZE_SPACE_ID = load_secret("ARIZE_SPACE_ID", required=False)

# Optional LangSmith switches if key exists
if LANGSMITH_API_KEY:
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "Capstone Project-Rajendra"


GROQ_API_KEY: loaded
LANGSMITH_API_KEY: loaded
ARIZE_API_KEY: loaded
ARIZE_SPACE_ID: loaded


## 3) Clone the Project Repository

Clone the training repository and Day 5 capstone folder.


In [3]:
!rm -rf Rajendra-Training
!git clone -q https://github.com/RajendraRajuk/Rajendra-Training.git
!ls Rajendra-Training/day5/capstone-project

 data   guides	 Ignore   README.md  'Sample Docs'


## 4) Imports and Folder Setup

In [4]:
from pathlib import Path
from typing import TypedDict, Optional, Dict, Any, Literal
import json
import sqlite3
from datetime import datetime, timezone

from pypdf import PdfReader
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

BASE_DIR = Path("Rajendra-Training/day5/capstone-project")
DATA_DIR = BASE_DIR / "data" / "pdfs"
SAMPLE_DIR = BASE_DIR / "Sample Docs"

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

DB_PATH = OUTPUT_DIR / "cease_documents.db"
ARCHIVE_PATH = OUTPUT_DIR / "irrelevant_archive.jsonl"
AUDIT_PATH = OUTPUT_DIR / "audit_log.jsonl"
HITL_QUEUE_PATH = OUTPUT_DIR / "hitl_queue.jsonl"

print("Data dir exists:", DATA_DIR.exists())
print("Sample dir exists:", SAMPLE_DIR.exists())


Data dir exists: True
Sample dir exists: True


## 5) Helper Functions

Utility functions used across loading, file handling, and logging.


In [5]:
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def append_jsonl(path: Path, payload: dict) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(payload, ensure_ascii=False) + "\n")

def read_pdf_text(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    text_parts = []
    for page in reader.pages:
        text_parts.append(page.extract_text() or "")
    return "\n".join(text_parts).strip()

def get_candidate_pdfs() -> list[Path]:
    files = []
    if DATA_DIR.exists():
        files.extend(sorted(DATA_DIR.glob("*.pdf")))
    if SAMPLE_DIR.exists():
        files.extend(sorted(SAMPLE_DIR.glob("*.pdf")))

    # Keep one copy when the same PDF exists in more than one folder.
    # Deduplicate by normalized file name first, then by content signature.
    unique_files = []
    seen_names = set()
    seen_signatures = set()

    for f in files:
        normalized_name = f.name.strip().lower()
        try:
            stat = f.stat()
            signature = (normalized_name, stat.st_size)
        except Exception:
            signature = (normalized_name, None)

        if normalized_name in seen_names or signature in seen_signatures:
            continue

        seen_names.add(normalized_name)
        seen_signatures.add(signature)
        unique_files.append(f)

    return unique_files


## 6) Database Setup

The database stores:
- date received
- document name
- extracted details

This matches the capstone requirement for **Cease** documents.


In [6]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute(
    '''
    CREATE TABLE IF NOT EXISTS cease_documents (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        date_received TEXT NOT NULL,
        document_name TEXT NOT NULL,
        classification TEXT NOT NULL,
        confidence REAL NOT NULL,
        customer_name TEXT,
        request_summary TEXT,
        extracted_details TEXT,
        processed_at TEXT NOT NULL
    )
    '''
)
conn.commit()

print("SQLite ready at:", DB_PATH)


SQLite ready at: outputs/cease_documents.db


## 7) LLM and Structured Output Schema

The classifier returns:
- `label`
- `confidence`
- `explanation`
- `customer_name`
- `request_summary`
- `key_details`

A confidence score lower than **0.60** is automatically converted to **Uncertain**.


In [7]:
class ClassificationResult(BaseModel):
    label: Literal["Cease", "Uncertain", "Irrelevant"] = Field(..., description="Final document class")
    confidence: float = Field(..., ge=0.0, le=1.0, description="Model confidence")
    explanation: str = Field(..., description="Short explanation of the decision")
    customer_name: str = Field(default="", description="Best effort extraction of customer/sender name")
    request_summary: str = Field(default="", description="Short summary of the request")
    key_details: str = Field(default="", description="Important extracted details from the document")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

structured_llm = llm.with_structured_output(ClassificationResult)


## 8) Agent Functions

The workflow is split into clear agent-style functions:
- document loader
- classifier
- database writer
- archive writer
- manual review
- audit logger


In [8]:

class WorkflowState(TypedDict, total=False):
    pdf_path: str
    document_name: str
    date_received: str
    raw_text: str
    classification: str
    confidence: float
    explanation: str
    customer_name: str
    request_summary: str
    key_details: str
    final_decision: str
    reviewer_name: str
    reviewer_notes: str
    route: str
    processed_at: str

def document_loader_agent(state: WorkflowState) -> WorkflowState:
    pdf_path = Path(state["pdf_path"])
    text = read_pdf_text(pdf_path)
    if not text:
        text = ""
    return {
        **state,
        "document_name": pdf_path.name,
        "raw_text": text,
        "date_received": utc_now_iso(),
    }

def classify_with_retry(text: str, max_retries: int = 3) -> ClassificationResult:
    prompt = f"""
You are classifying a PDF document for a cease and desist processing workflow.

Return one of only these labels:
- Cease
- Uncertain
- Irrelevant

Rules:
- If the document is clearly a valid request to stop communication, choose Cease.
- If the document is clearly unrelated, choose Irrelevant.
- If the content is incomplete, unclear, weak, or ambiguous, choose Uncertain.
- If confidence is below 0.60, still return your best label and confidence. Downstream logic will convert it to Uncertain.

Extract best-effort fields:
- customer_name
- request_summary
- key_details

Document text:
{text[:12000]}
"""
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            return structured_llm.invoke(prompt)
        except Exception as e:
            last_error = e
            print(f"Retry {attempt}/{max_retries} failed: {e}")
    raise RuntimeError(f"Classifier failed after {max_retries} retries: {last_error}")

def classification_agent(state: WorkflowState) -> WorkflowState:
    text = state.get("raw_text", "").strip()

    if not text:
        # Empty extraction should go for review and must still set final_decision
        return {
            **state,
            "classification": "Uncertain",
            "confidence": 0.0,
            "explanation": "PDF text extraction returned empty content.",
            "customer_name": "",
            "request_summary": "",
            "key_details": "",
            "final_decision": "Uncertain"
        }

    try:
        result = classify_with_retry(text)
        raw_label = result.label
        final_label = raw_label if float(result.confidence) >= 0.60 else "Uncertain"

        return {
            **state,
            "classification": raw_label,
            "confidence": float(result.confidence),
            "explanation": result.explanation,
            "customer_name": result.customer_name,
            "request_summary": result.request_summary,
            "key_details": result.key_details,
            "final_decision": final_label
        }
    except Exception as e:
        # Safe fallback: do not stop the workflow if classification fails
        return {
            **state,
            "classification": "Uncertain",
            "confidence": 0.0,
            "explanation": f"Classification failed and was sent for review: {e}",
            "customer_name": "",
            "request_summary": "",
            "key_details": "",
            "final_decision": "Uncertain"
        }

def router_agent(state: WorkflowState) -> str:
    decision = state.get("final_decision")

    # Defensive fallback in case a state reaches router without final_decision
    if not decision:
        decision = state.get("classification", "Uncertain")
        if state.get("confidence", 0.0) < 0.60:
            decision = "Uncertain"

    if decision == "Cease":
        return "database"
    if decision == "Irrelevant":
        return "archive"
    return "hitl"

def database_agent(state: WorkflowState) -> WorkflowState:
    payload = {
        "date_received": state["date_received"],
        "document_name": state["document_name"],
        "classification": "Cease",
        "confidence": state.get("confidence", 0.0),
        "customer_name": state.get("customer_name", ""),
        "request_summary": state.get("request_summary", ""),
        "extracted_details": state.get("key_details", ""),
        "processed_at": utc_now_iso()
    }
    cur.execute(
        """
        INSERT INTO cease_documents
        (date_received, document_name, classification, confidence, customer_name, request_summary, extracted_details, processed_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            payload["date_received"],
            payload["document_name"],
            payload["classification"],
            payload["confidence"],
            payload["customer_name"],
            payload["request_summary"],
            payload["extracted_details"],
            payload["processed_at"],
        )
    )
    conn.commit()
    return {
        **state,
        "final_decision": "Cease",
        "route": "database",
        "processed_at": payload["processed_at"]
    }

def archive_agent(state: WorkflowState) -> WorkflowState:
    payload = {
        "date_received": state["date_received"],
        "document_name": state["document_name"],
        "classification": "Irrelevant",
        "confidence": state.get("confidence", 0.0),
        "processed_at": utc_now_iso()
    }
    append_jsonl(ARCHIVE_PATH, payload)
    return {
        **state,
        "final_decision": "Irrelevant",
        "route": "archive",
        "processed_at": payload["processed_at"]
    }

# Optional preset review map for repeatable testing
PRESET_HITL_REVIEW = {
    # "file_name.pdf": {
    #     "decision": "approved",   # approved -> Cease, rejected -> Irrelevant
    #     "reviewer_name": "Rajendra",
    #     "reviewer_notes": "Manual review completed"
    # }
}

INTERACTIVE_HITL = True

def normalize_human_decision(value: str) -> str:
    value = (value or "").strip().lower()
    if value in {"approved", "approve", "a", "cease"}:
        return "Cease"
    if value in {"rejected", "reject", "r", "irrelevant"}:
        return "Irrelevant"
    return "Irrelevant"

def hitl_agent(state: WorkflowState) -> WorkflowState:
    file_name = state["document_name"]

    if file_name in PRESET_HITL_REVIEW:
        review = PRESET_HITL_REVIEW[file_name]
        decision = normalize_human_decision(review.get("decision", "rejected"))
        reviewer_name = review.get("reviewer_name", "Rajendra")
        reviewer_notes = review.get("reviewer_notes", "Preset manual review")
    elif INTERACTIVE_HITL:
        print("\n--- Manual Review Required ---")
        print("Document:", file_name)
        print("Model label:", state.get("classification"))
        print("Confidence:", state.get("confidence"))
        print("Explanation:", state.get("explanation"))
        print("Summary:", state.get("request_summary"))
        print("Choose human decision:")
        print("  approved -> treat as Cease")
        print("  rejected -> treat as Irrelevant")
        print("------------------------------")

        human_decision = input("Enter human decision (approved/rejected): ").strip()
        decision = normalize_human_decision(human_decision)

        reviewer_name = input("Reviewer name: ").strip() or "Rajendra"
        reviewer_notes = input("Reviewer notes: ").strip() or "Manual review completed."
    else:
        decision = "Irrelevant"
        reviewer_name = "Rajendra"
        reviewer_notes = "Non-interactive fallback used during batch execution."

    updated = {
        **state,
        "final_decision": decision,
        "reviewer_name": reviewer_name,
        "reviewer_notes": reviewer_notes
    }

    if decision == "Cease":
        updated = database_agent(updated)
    else:
        updated = archive_agent(updated)

    append_jsonl(HITL_QUEUE_PATH, {
        "document_name": file_name,
        "model_label": state.get("classification"),
        "human_decision": "approved" if decision == "Cease" else "rejected",
        "final_decision": decision,
        "reviewer_name": reviewer_name,
        "reviewer_notes": reviewer_notes,
        "processed_at": updated.get("processed_at", utc_now_iso())
    })

    return updated

def audit_agent(state: WorkflowState) -> WorkflowState:
    append_jsonl(AUDIT_PATH, {
        "date_received": state.get("date_received"),
        "document_name": state.get("document_name"),
        "classification": state.get("classification"),
        "confidence": state.get("confidence"),
        "explanation": state.get("explanation"),
        "final_decision": state.get("final_decision"),
        "route": state.get("route"),
        "reviewer_name": state.get("reviewer_name", ""),
        "reviewer_notes": state.get("reviewer_notes", ""),
        "processed_at": state.get("processed_at", utc_now_iso())
    })
    return state


## 9) Build the LangGraph Workflow


In [9]:
graph = StateGraph(WorkflowState)

graph.add_node("document_loader", document_loader_agent)
graph.add_node("classifier", classification_agent)
graph.add_node("database", database_agent)
graph.add_node("archive", archive_agent)
graph.add_node("hitl", hitl_agent)
graph.add_node("audit", audit_agent)

graph.set_entry_point("document_loader")
graph.add_edge("document_loader", "classifier")

graph.add_conditional_edges(
    "classifier",
    router_agent,
    {
        "database": "database",
        "archive": "archive",
        "hitl": "hitl"
    }
)

graph.add_edge("database", "audit")
graph.add_edge("archive", "audit")
graph.add_edge("hitl", "audit")
graph.add_edge("audit", END)

app = graph.compile()
print("LangGraph workflow compiled successfully.")


LangGraph workflow compiled successfully.


## 10) Test Configuration

Supported values:
- `one`
- `two`
- `all`

`INTERACTIVE_HITL` is enabled automatically for `one` and `two`.
For `all`, the notebook switches to non-interactive fallback unless you add a preset review entry.


In [11]:

TEST_MODE = "all"   # change to: one / two / all
INTERACTIVE_HITL = TEST_MODE in {"one", "two","all"}

all_pdfs = get_candidate_pdfs()

if TEST_MODE == "one":
    selected_pdfs = all_pdfs[:1]
elif TEST_MODE == "two":
    selected_pdfs = all_pdfs[:2]
else:
    selected_pdfs = all_pdfs

print("Selected PDFs:")
for item in selected_pdfs:
    print("-", item)

print("\nInteractive HITL:", INTERACTIVE_HITL)


Selected PDFs:
- Rajendra-Training/day5/capstone-project/data/pdfs/LOA2.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/LOA3.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/LOA4.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/LOA5.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/LOA6.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/LOA7.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/LOA8.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/LOA9.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/LoA1.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/bw_doc_1.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/bw_doc_2.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/bw_doc_3.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/bw_doc_4.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/bw_doc_5.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/notice_1.pdf
- Rajendra-Training/day5/capstone-project/data

## 11) Execute the Pipeline

In [13]:

results = []

for pdf_path in selected_pdfs:
    try:
        initial_state: WorkflowState = {"pdf_path": str(pdf_path)}
        final_state = app.invoke(initial_state)
        results.append(final_state)

        print("\nProcessed:", final_state.get("document_name"))
        print("Model label:", final_state.get("classification"))
        print("Confidence:", final_state.get("confidence"))
        print("Final decision:", final_state.get("final_decision"))
        print("Route:", final_state.get("route"))
    except Exception as e:
        error_record = {
            "document_name": pdf_path.name,
            "error": str(e),
            "processed_at": utc_now_iso()
        }
        append_jsonl(OUTPUT_DIR / "processing_errors.jsonl", error_record)
        print(f"\nProcessing failed for {pdf_path.name}: {e}")



Processed: LOA2.pdf
Model label: Cease
Confidence: 0.95
Final decision: Cease
Route: database

Processed: LOA3.pdf
Model label: Cease
Confidence: 0.95
Final decision: Cease
Route: database

Processed: LOA4.pdf
Model label: Cease
Confidence: 0.95
Final decision: Cease
Route: database

Processed: LOA5.pdf
Model label: Cease
Confidence: 0.95
Final decision: Cease
Route: database

Processed: LOA6.pdf
Model label: Cease
Confidence: 0.95
Final decision: Cease
Route: database

Processed: LOA7.pdf
Model label: Irrelevant
Confidence: 0.8
Final decision: Irrelevant
Route: archive

Processed: LOA8.pdf
Model label: Cease
Confidence: 0.8
Final decision: Cease
Route: database

Processed: LOA9.pdf
Model label: Cease
Confidence: 0.85
Final decision: Cease
Route: database

Processed: LoA1.pdf
Model label: Cease
Confidence: 0.95
Final decision: Cease
Route: database

--- Manual Review Required ---
Document: bw_doc_1.pdf
Model label: Uncertain
Confidence: 0.0
Explanation: PDF text extraction returned em

## 12) Review Outputs

Use this section to inspect the saved outputs after a run.


In [14]:
print("DB file:", DB_PATH)
print("Archive file:", ARCHIVE_PATH)
print("Audit file:", AUDIT_PATH)
print("HITL file:", HITL_QUEUE_PATH)

# Show DB rows
rows = cur.execute(
    "SELECT date_received, document_name, customer_name, request_summary, extracted_details FROM cease_documents"
).fetchall()

print("\nCease documents stored in SQLite:")
for row in rows:
    print(row)


DB file: outputs/cease_documents.db
Archive file: outputs/irrelevant_archive.jsonl
Audit file: outputs/audit_log.jsonl
HITL file: outputs/hitl_queue.jsonl

Cease documents stored in SQLite:
('2026-03-25T19:21:36.421210+00:00', 'LOA2.pdf', 'LOVETTA CANNUNZIATA', 'Cease and desist all communications regarding the account', 'Limited Power of Attorney granted to LAW OFFICES OF DONALD A. GREEN, APLC')
('2026-03-25T19:21:36.992797+00:00', 'LOA3.pdf', 'BELLINA DAVANZO AMOEDO, RADCLIFF', 'Request to cease and desist all communications regarding the account and grant a Limited Power of Attorney to Five Lakes Law Group PLLC', 'Account/Reference#: _________ _, Limited Power of Attorney granted to Five Lakes Law Group PLLC')
('2026-03-25T19:21:37.942048+00:00', 'LOA4.pdf', 'DAVANZO,BELLINA AMOEDO,RADCLIFF', 'Cease and desist all communications regarding the account', 'Limited Power of Attorney granted to Five Lakes Law Group PLLC')
('2026-03-25T19:21:38.711011+00:00', 'LOA5.pdf', 'RANDLE L WIDHALM

In [15]:
def preview_jsonl(path: Path, limit: int = 5):
    print(f"\nPreview -> {path}")
    if not path.exists():
        print("File not found.")
        return
    with open(path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            if idx >= limit:
                break
            print(json.loads(line))

preview_jsonl(ARCHIVE_PATH)
preview_jsonl(AUDIT_PATH)
preview_jsonl(HITL_QUEUE_PATH)



Preview -> outputs/irrelevant_archive.jsonl
{'date_received': '2026-03-25T19:21:41.973317+00:00', 'document_name': 'LOA7.pdf', 'classification': 'Irrelevant', 'confidence': 0.8, 'processed_at': '2026-03-25T19:21:42.649640+00:00'}
{'date_received': '2026-03-25T19:21:48.271836+00:00', 'document_name': 'bw_doc_1.pdf', 'classification': 'Irrelevant', 'confidence': 0.0, 'processed_at': '2026-03-25T19:23:42.224237+00:00'}
{'date_received': '2026-03-25T19:23:42.231513+00:00', 'document_name': 'bw_doc_2.pdf', 'classification': 'Irrelevant', 'confidence': 0.0, 'processed_at': '2026-03-25T19:24:00.941266+00:00'}
{'date_received': '2026-03-25T19:24:00.948039+00:00', 'document_name': 'bw_doc_3.pdf', 'classification': 'Irrelevant', 'confidence': 0.0, 'processed_at': '2026-03-25T19:24:25.266776+00:00'}
{'date_received': '2026-03-25T19:24:25.274469+00:00', 'document_name': 'bw_doc_4.pdf', 'classification': 'Irrelevant', 'confidence': 0.0, 'processed_at': '2026-03-25T19:24:33.961103+00:00'}

Preview 

## 13) Notes

Current coverage:
- multiple agents
- LangGraph workflow
- manual review path
- SQLite interaction
- flat-file archiving
- audit trail

Small additions used in the flow:
- confidence threshold (`< 0.60 -> Uncertain`)
- 3 retry attempts for structured output
- test mode for one / two / all PDFs
